# Regresión Logística - Clasificación Binaria desde Cero

Bienvenido al segundo notebook de algoritmos supervisados. La Regresión Logística es el algoritmo fundamental para **clasificación binaria** y la extensión natural de la regresión lineal.

Al finalizar este notebook, serás capaz de:

* Comprender la función sigmoide y su papel en clasificación
* Implementar regresión logística desde cero usando NumPy
* Entender Binary Cross-Entropy como función de costo
* Aplicar Gradient Descent para problemas de clasificación
* Interpretar probabilidades y fronteras de decisión
* Evaluar modelos con métricas de clasificación (Accuracy, Precision, Recall, F1)
* Comprender la matriz de confusión y sus implicaciones

**¿Por qué Regresión Logística?**

1. **Fundamento de clasificación**: Base para entender algoritmos más complejos
2. **Probabilidades**: Retorna probabilidades interpretables, no solo clases
3. **Eficiente**: Rápido de entrenar y hacer predicciones
4. **Baseline poderoso**: Excelente punto de partida antes de modelos complejos
5. **Aplicaciones reales**: Detección de spam, diagnóstico médico, credit scoring

## Nota Importante sobre los Ejercicios

Antes de comenzar con los ejercicios, ten en cuenta lo siguiente:

1. NO agregues declaraciones `print` adicionales en las funciones graduadas
2. NO agregues celdas de código adicionales entre los ejercicios
3. NO cambies los parámetros de las funciones
4. Implementa usando NumPy (operaciones vectorizadas, sin bucles cuando sea posible)
5. NO cambies el código de las pruebas automáticas

Si experimentas errores al ejecutar las pruebas, primero verifica estos puntos antes de buscar ayuda.

<a name='1'></a>
## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Teoría de Regresión Logística](#2)
- [3 - Implementación desde Cero](#3)
- [4 - Clasificación Binaria](#4)
  - [Ejercicio 1](#ex01)
- [5 - Métricas de Evaluación](#5)
  - [Ejercicio 2](#ex02)
- [6 - Referencias](#6)

<a name='1'></a>
## 1 - Paquetes

Ejecuta la siguiente celda para importar los paquetes que usarás en este notebook:

* **NumPy**: Operaciones numéricas y álgebra lineal
* **Matplotlib**: Visualización de resultados y gráficos
* **Scikit-learn**: Generación de datos sintéticos
* **Testing utilities**: Verificación automática de ejercicios

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from sklearn.datasets import make_classification

# Configurar matplotlib inline
%matplotlib inline

# Agregar el directorio raíz al path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Importar utilidades de testing
from utils.testing_utils import print_success, print_error, print_info
from tests.supervisados.test_02_regresion_logistica import (
    test_ejercicio_1_sigmoid,
    test_ejercicio_2_metricas
)

print("✅ Paquetes importados correctamente")
print(f"📦 NumPy version: {np.__version__}")

<a name='2'></a>
## 2 - Teoría de Regresión Logística

La regresión logística es un algoritmo de **clasificación** (no regresión, a pesar del nombre) que predice la probabilidad de que una muestra pertenezca a una clase.

### 2.1 - La Función Sigmoide

La función sigmoide (también llamada logística) transforma cualquier valor real en un rango [0, 1]:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Propiedades importantes:**
- $\sigma(0) = 0.5$
- $\lim_{z \to \infty} \sigma(z) = 1$
- $\lim_{z \to -\infty} \sigma(z) = 0$
- Derivada: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$

### 2.2 - El Modelo de Regresión Logística

El modelo combina una transformación lineal con la función sigmoide:

$$z = \mathbf{w}^T \mathbf{x} + b$$

$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

donde:
- $z$ = combinación lineal de features (logit)
- $\hat{y}$ = probabilidad predicha de clase 1, es decir $P(y=1 | \mathbf{x})$
- Decisión: si $\hat{y} \geq 0.5$, predecir clase 1, sino clase 0

### 2.3 - Binary Cross-Entropy Loss

Para clasificación binaria, usamos Binary Cross-Entropy (también llamada Log Loss):

$$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1-y^{(i)}) \log(1-\hat{y}^{(i)}) \right]$$

**Interpretación:**
- Si $y^{(i)} = 1$: queremos maximizar $\log(\hat{y}^{(i)})$, es decir, que $\hat{y}^{(i)} \to 1$
- Si $y^{(i)} = 0$: queremos maximizar $\log(1 - \hat{y}^{(i)})$, es decir, que $\hat{y}^{(i)} \to 0$

### 2.4 - Gradientes para Gradient Descent

Las derivadas parciales son (¡idénticas a regresión lineal!):

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

Aunque las fórmulas son iguales, **$\hat{y}$ es diferente**: en regresión lineal es $\mathbf{w}^T \mathbf{x} + b$, aquí es $\sigma(\mathbf{w}^T \mathbf{x} + b)$.

### 2.5 - Visualización de la Función Sigmoide

Vamos a graficar la función sigmoide para entender su comportamiento:

In [ ]:
# Visualizar función sigmoide
z = np.linspace(-10, 10, 100)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(10, 6))
plt.plot(z, sigmoid, linewidth=2, label='σ(z)')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Threshold = 0.5')
plt.axvline(x=0, color='g', linestyle='--', alpha=0.5, label='z = 0')
plt.xlabel('z (logit)')
plt.ylabel('σ(z) (probabilidad)')
plt.title('Función Sigmoide')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"σ(-5) = {1/(1+np.exp(5)):.4f}")
print(f"σ(0)  = {1/(1+np.exp(0)):.4f}")
print(f"σ(5)  = {1/(1+np.exp(-5)):.4f}")

<a name='3'></a>
## 3 - Implementación desde Cero

Vamos a implementar una clase completa de Regresión Logística con Gradient Descent.

### 3.1 - Estructura de la Clase

Nuestra clase tendrá:
* `_sigmoid(z)`: Calcula la función sigmoide
* `fit(X, y)`: Entrena el modelo usando Gradient Descent
* `predict_proba(X)`: Retorna probabilidades
* `predict(X)`: Retorna clases predichas (0 o 1)
* `score(X, y)`: Calcula accuracy

In [ ]:
class RegresionLogistica:
    """
    Regresión Logística implementada desde cero con Gradient Descent.
    
    Parámetros:
    -----------
    learning_rate : float
        Tasa de aprendizaje (alpha)
    n_iterations : int
        Número de iteraciones para gradient descent
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def _sigmoid(self, z):
        """Función sigmoide"""
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        """
        Entrena el modelo usando Gradient Descent.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Features de entrenamiento
        y : array-like, shape (n_samples,)
            Target de entrenamiento (0 o 1)
        """
        X = np.array(X)
        y = np.array(y)
        
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        n_samples, n_features = X.shape
        
        # Inicializar parámetros
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient Descent
        for i in range(self.n_iterations):
            # Forward pass
            z = np.dot(X, self.weights) + self.bias
            y_pred = self._sigmoid(z)
            
            # Calcular gradientes
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Actualizar parámetros
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Calcular loss (Binary Cross-Entropy)
            loss = self._binary_cross_entropy(y, y_pred)
            self.loss_history.append(loss)
            
            if (i + 1) % 100 == 0:
                print(f"Iteración {i+1}/{self.n_iterations}, Loss: {loss:.4f}")
        
        return self
    
    def _binary_cross_entropy(self, y_true, y_pred):
        """Binary Cross-Entropy Loss"""
        # Evitar log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def predict_proba(self, X):
        """Predice probabilidades"""
        X = np.array(X)
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        z = np.dot(X, self.weights) + self.bias
        return self._sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        """Predice clases (0 o 1)"""
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase RegresionLogistica definida correctamente")

<a name='4'></a>
## 4 - Clasificación Binaria

Vamos a aplicar regresión logística a un problema de clasificación binaria.

### 4.1 - Generar Datos Sintéticos

Usamos `make_classification` de scikit-learn para crear un dataset balanceado:

In [ ]:
# Generar datos sintéticos
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                          n_informative=2, n_clusters_per_class=1, 
                          random_state=42)

# Visualizar
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Clase 0', edgecolors='k', alpha=0.7)
plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Clase 1', edgecolors='k', alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos de Clasificación Binaria')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total de muestras: {len(X)}")
print(f"Clase 0: {np.sum(y==0)} muestras ({np.sum(y==0)/len(y)*100:.1f}%)")
print(f"Clase 1: {np.sum(y==1)} muestras ({np.sum(y==1)/len(y)*100:.1f}%)")

### 4.2 - Entrenar el Modelo

Entrenamos nuestro modelo de regresión logística:

In [ ]:
# Entrenar modelo
modelo = RegresionLogistica(learning_rate=0.1, n_iterations=1000)
modelo.fit(X, y)

# Evaluar
accuracy = modelo.score(X, y)
print(f"\nAccuracy: {accuracy:.4f}")

# Mostrar algunos ejemplos de predicciones
print("\nEjemplos de predicciones:")
probas = modelo.predict_proba(X[:5])
preds = modelo.predict(X[:5])
for i in range(5):
    print(f"Muestra {i}: P(y=1) = {probas[i]:.4f}, Predicción = {preds[i]}, Real = {y[i]}")

### 4.3 - Visualizar Frontera de Decisión

La frontera de decisión es donde $P(y=1) = 0.5$, es decir, donde $\mathbf{w}^T \mathbf{x} + b = 0$:

In [ ]:
# Visualizar frontera de decisión
def plot_decision_boundary(X, y, model):
    """Grafica la frontera de decisión"""
    # Crear grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predecir probabilidades para cada punto del grid
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Graficar
    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
    plt.colorbar(label='P(y=1)')
    
    # Frontera de decisión (P = 0.5)
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    
    # Datos
    plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Clase 0', edgecolors='k')
    plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Clase 1', edgecolors='k')
    
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Frontera de Decisión - Regresión Logística')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_decision_boundary(X, y, modelo)

### 4.4 - Curva de Aprendizaje

Observamos cómo disminuye el Binary Cross-Entropy durante el entrenamiento:

In [ ]:
# Curva de aprendizaje
plt.figure(figsize=(10, 6))
plt.plot(modelo.loss_history, linewidth=2)
plt.xlabel('Iteración')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Curva de Aprendizaje')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Loss inicial: {modelo.loss_history[0]:.4f}")
print(f"Loss final: {modelo.loss_history[-1]:.4f}")
print(f"Reducción: {(1 - modelo.loss_history[-1]/modelo.loss_history[0]) * 100:.2f}%")

<a name='6'></a>
## 6 - Referencias

### Papers Fundamentales

1. **Cox, D. R.** (1958). "The regression analysis of binary sequences." *Journal of the Royal Statistical Society: Series B (Methodological)*, 20(2), 215-232.
   - Introducción formal de regresión logística

2. **Hosmer, D. W., & Lemeshow, S.** (2000). *Applied Logistic Regression*. Wiley.
   - Texto clásico sobre regresión logística aplicada

3. **Walker, S. H., & Duncan, D. B.** (1967). "Estimation of the probability of an event as a function of several independent variables." *Biometrika*, 54(1-2), 167-179.
   - Estimación máxima verosimilitud para regresión logística

### Recursos Adicionales

4. **Bishop, C. M.** (2006). *Pattern Recognition and Machine Learning*. Springer. Chapter 4: Linear Models for Classification.
   - Tratamiento probabilístico de clasificación lineal

5. **Murphy, K. P.** (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press. Chapter 8: Logistic Regression.
   - Perspectiva bayesiana y probabilística

6. **Hastie, T., Tibshirani, R., & Friedman, J.** (2009). *The Elements of Statistical Learning*. Springer. Chapter 4: Linear Methods for Classification.
   - Tratamiento estadístico riguroso

### Recursos Online

7. **Andrew Ng's Machine Learning Course** - Coursera
   - https://www.coursera.org/learn/machine-learning
   - Excelente introducción con ejemplos prácticos

8. **Scikit-learn Documentation: Logistic Regression**
   - https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
   - Implementación optimizada y variantes (L1, L2, Elastic Net)

9. **StatQuest: Logistic Regression**
   - https://www.youtube.com/watch?v=yIYKR4sgzI8
   - Explicación visual e intuitiva

### Extensiones y Variantes

10. **Multinomial Logistic Regression** (Softmax): Para clasificación multiclase
11. **Regularized Logistic Regression**: L1 (Lasso), L2 (Ridge) para prevenir overfitting
12. **Weighted Logistic Regression**: Para datasets desbalanceados
13. **Ordinal Logistic Regression**: Para variables ordinales

### Métricas de Evaluación

14. **Precision-Recall Curve**: Útil para datasets desbalanceados
15. **ROC Curve y AUC**: Mide discriminación del clasificador
16. **Calibration Plots**: Verifica si las probabilidades están bien calibradas

## 📘 Resumen y Aplicaciones en ML

<div style="background-color: #e7f3fe; padding: 20px; border-left: 6px solid #2196F3; margin: 20px 0;">

**Conceptos Clave Aprendidos:**

1. **Función Sigmoide**: Transforma logits en probabilidades interpretables [0, 1]
2. **Binary Cross-Entropy**: Función de costo adecuada para clasificación binaria
3. **Probabilidades**: A diferencia de clasificadores duros, retorna probabilidades
4. **Frontera de Decisión**: Línea donde $P(y=1) = 0.5$, definida por $\mathbf{w}^T \mathbf{x} + b = 0$
5. **Métricas**: Precision, Recall, F1 complementan Accuracy para evaluación completa

**Relevancia en Machine Learning:**

- **Fundamento de redes neuronales**: La sigmoide es una función de activación común
- **Clasificación multiclase**: Se extiende a Softmax Regression para múltiples clases
- **Calibración de probabilidades**: Las probabilidades son útiles para toma de decisiones
- **Transfer learning**: Capa final de muchos modelos de clasificación
- **Aplicaciones reales**: 
  - Detección de spam (spam vs no spam)
  - Diagnóstico médico (enfermo vs sano)
  - Credit scoring (aprobar vs rechazar préstamo)
  - Click prediction (click vs no click)

**¿Cuándo usar Regresión Logística?**

✅ **Usar cuando:**
- Necesitas probabilidades interpretables
- El problema es linealmente separable o casi
- Quieres un baseline rápido y eficiente
- La interpretabilidad es importante

❌ **No usar cuando:**
- Los datos tienen relaciones no lineales complejas
- Hay muchas interacciones entre features
- El problema requiere fronteras de decisión complejas

</div>

In [ ]:
# GRADED FUNCTION: compute_precision

def compute_precision(y_true, y_pred):
    """
    Calcula la métrica Precision.
    
    Parámetros
    ----------
    y_true : ndarray
        Etiquetas verdaderas (0 o 1)
    y_pred : ndarray
        Etiquetas predichas (0 o 1)
    
    Retorna
    -------
    float
        Valor de Precision en rango [0, 1]
    
    Ejemplo
    -------
    >>> y_true = np.array([1, 0, 1, 1, 0])
    >>> y_pred = np.array([1, 0, 1, 0, 0])
    >>> compute_precision(y_true, y_pred)
    1.0
    """
    
    ### YOUR CODE STARTS HERE ###
    # Paso 1: Calcular TP (True Positives)
    TP = None
    
    # Paso 2: Calcular FP (False Positives)
    FP = None
    
    # Paso 3: Calcular Precision
    precision = None
    ### YOUR CODE ENDS HERE ###
    
    return precision

# Prueba tu implementación
y_test = np.array([1, 0, 1, 1, 0, 1, 0, 0])
y_pred_test = np.array([1, 0, 1, 0, 0, 1, 1, 0])

resultado = compute_precision(y_test, y_pred_test)
print(f"Precision: {resultado:.4f}")

# Verificar con test automático
verificar_precision = test_ejercicio_2_metricas()
verificar_precision(compute_precision)

<a name='ex02'></a>
### Ejercicio 2: Calcular Precision

Implementa una función que calcule la métrica Precision.

**Instrucciones:**
- Precision = TP / (TP + FP)
- Si no hay predicciones positivas, retornar 0.0
- Usa operaciones vectorizadas de NumPy

In [ ]:
def calcular_metricas(y_true, y_pred):
    """Calcula métricas de clasificación"""
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': np.array([[TN, FP], [FN, TP]])
    }

y_pred = modelo.predict(X)
metricas = calcular_metricas(y, y_pred)

print("=" * 50)
print("MÉTRICAS DE EVALUACIÓN")
print("=" * 50)
print(f"Accuracy:  {metricas['accuracy']:.4f}")
print(f"Precision: {metricas['precision']:.4f}")
print(f"Recall:    {metricas['recall']:.4f}")
print(f"F1-Score:  {metricas['f1']:.4f}")
print(f"\nMatriz de Confusión:")
print("                Predicho")
print("              0       1")
print(f"Real    0  [{metricas['confusion_matrix'][0,0]:4d}  {metricas['confusion_matrix'][0,1]:4d}]")
print(f"        1  [{metricas['confusion_matrix'][1,0]:4d}  {metricas['confusion_matrix'][1,1]:4d}]")

### 5.3 - Calcular Métricas

Calculamos las métricas para nuestro modelo:

<a name='5'></a>
## 5 - Métricas de Evaluación

Para clasificación, accuracy no siempre es suficiente. Necesitamos entender diferentes tipos de errores.

### 5.1 - Matriz de Confusión

La matriz de confusión muestra los 4 tipos de resultados posibles:

$$
\begin{bmatrix}
\text{TN} & \text{FP} \\
\text{FN} & \text{TP}
\end{bmatrix}
$$

donde:
- **TP** (True Positive): Predijimos 1, real es 1 ✅
- **TN** (True Negative): Predijimos 0, real es 0 ✅
- **FP** (False Positive): Predijimos 1, real es 0 ❌ (Error Tipo I)
- **FN** (False Negative): Predijimos 0, real es 1 ❌ (Error Tipo II)

### 5.2 - Métricas Derivadas

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP}$$ 
*De las predicciones positivas, ¿cuántas son correctas?*

$$\text{Recall (Sensitivity)} = \frac{TP}{TP + FN}$$
*De los casos realmente positivos, ¿cuántos detectamos?*

$$\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
*Media armónica de Precision y Recall*

In [ ]:
# GRADED FUNCTION: sigmoid

def sigmoid(z):
    """
    Calcula la función sigmoide.
    
    Parámetros
    ----------
    z : ndarray or float
        Valor(es) de entrada
    
    Retorna
    -------
    ndarray or float
        Valor(es) de salida en rango [0, 1]
    
    Ejemplo
    -------
    >>> sigmoid(0)
    0.5
    >>> sigmoid(np.array([-1000, 0, 1000]))
    array([0., 0.5, 1.])
    """
    
    ### YOUR CODE STARTS HERE ###
    result = None
    ### YOUR CODE ENDS HERE ###
    
    return result

# Prueba tu implementación
test_values = np.array([-5, 0, 5])
resultado = sigmoid(test_values)
print(f"sigmoid({test_values}) = {resultado}")

# Verificar con test automático
verificar_sigmoid = test_ejercicio_1_sigmoid()
verificar_sigmoid(sigmoid)

<a name='ex01'></a>
### Ejercicio 1: Implementar Función Sigmoide

Implementa la función sigmoide que transforma valores en probabilidades.

**Instrucciones:**
- Usa operaciones vectorizadas de NumPy
- Maneja valores extremos (evita overflow con `np.clip`)
- Retorna valores en el rango [0, 1]